# NB-602: Advanced RAG Techniques

Improve RAG with advanced retrieval methods.

## Learning Objectives
- Hybrid search (dense + sparse)
- Re-ranking with cross-encoders
- Query expansion
- Context window optimization

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np

print("Libraries imported")

## 1. Hybrid Search

In [ ]:
class HybridRetriever:
    def __init__(self, documents, embedder):
        self.documents = documents
        self.embedder = embedder
        self.dense_embeddings = embedder.encode(documents)

    def hybrid_search(self, query, alpha=0.5, top_k=5):
        """Combine dense and sparse search."""

        # TODO: Dense search
        query_emb = self.embedder.encode([query])[0]
        dense_scores = np.dot(self.dense_embeddings, query_emb)

        # TODO: Sparse search (keyword matching)
        query_words = set(query.lower().split())
        sparse_scores = []
        for doc in self.documents:
            doc_words = set(doc.lower().split())
            overlap = len(query_words & doc_words)
            sparse_scores.append(overlap)

        sparse_scores = np.array(sparse_scores)

        # TODO: Normalize and combine
        dense_scores = (dense_scores - dense_scores.min()) / (dense_scores.max() - dense_scores.min() + 1e-8)
        sparse_scores = (sparse_scores - sparse_scores.min()) / (sparse_scores.max() - sparse_scores.min() + 1e-8)

        combined = alpha * dense_scores + (1 - alpha) * sparse_scores

        # Get top-k
        top_indices = combined.argsort()[-top_k:][::-1]

        return [(self.documents[i], combined[i]) for i in top_indices]

# Test
docs = ["Machine learning uses neural networks", "Deep learning is a subset of ML"]
retriever = HybridRetriever(docs, SentenceTransformer('all-MiniLM-L6-v2'))
results = retriever.hybrid_search("neural networks")
for doc, score in results:
    print(f"[{score:.3f}] {doc}")

## 2. Re-ranking with Cross-Encoder

In [ ]:
class RerankingRetriever:
    def __init__(self, documents, embedder, reranker):
        self.documents = documents
        self.embedder = embedder
        self.reranker = reranker
        self.embeddings = embedder.encode(documents)

    def retrieve_with_rerank(self, query, top_k=10, rerank_top=3):
        """Retrieve with cross-encoder re-ranking."""

        # TODO: Initial retrieval
        query_emb = self.embedder.encode([query])[0]
        scores = np.dot(self.embeddings, query_emb)
        top_indices = scores.argsort()[-top_k:][::-1]

        candidates = [(self.documents[i], scores[i]) for i in top_indices]

        # TODO: Re-rank with cross-encoder
        pairs = [[query, doc] for doc, _ in candidates]
        rerank_scores = self.reranker.predict(pairs)

        # Sort by rerank scores
        reranked = list(zip(candidates, rerank_scores))
        reranked.sort(key=lambda x: x[1], reverse=True)

        return [(doc, score) for (doc, _), score in reranked[:rerank_top]]

print("Reranking retriever defined")

## 3. Query Expansion

In [ ]:
class QueryExpander:
    def __init__(self, llm):
        self.llm = llm

    def expand(self, query, n_expansions=3):
        """Generate query expansions."""

        prompt = f"Generate {n_expansions} alternative ways to ask: {query}\n"
        prompt += "List each on a new line."

        # TODO: Use LLM to expand
        response = self.llm(prompt)

        expansions = [query]  # Include original
        # Parse response (simplified)
        lines = response.split("\n")
        for line in lines[:n_expansions]:
            if line.strip():
                expansions.append(line.strip())

        return expansions

print("Query expander defined")

## Exercise

1. Implement recursive retrieval
2. Add metadata filtering
3. Implement multi-vector retrieval